In [2]:
%pwd


'd:\\Arambh\\project\\kedney\\Kidney-Disease-Classification\\research'

In [3]:
import os
os.chdir("../")

In [11]:
%pwd

'd:\\Arambh\\project\\kedney\\Kidney-Disease-Classification'

In [12]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class TrainingConfig:
    root_dir: Path
    trained_model_path: Path
    updated_base_model_path: Path
    training_data: Path
    params_epochs: int
    params_batch_size: int
    params_is_augmentation: bool
    params_image_size: list


In [13]:
from cnnClassifier.constants import *
from cnnClassifier.utils.common import read_yaml,create_directorries
import tensorflow as tf

In [14]:
class ConfigurationManger:
    def __init__(
        self ,config_file_path = CONFIG_FILE_PATH,
        params_file_path = PARAMS_FILE_PATH):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)
        # logger(f"Path : {CONFIG_FILE_PATH}")
        # print("fdsfdsfdsfdsf")
        create_directorries([self.config.artifacts_root])

    def get_training_config(self)->TrainingConfig:
        training = self.config.training
        prepare_base_model = self.config.prepare_base_model
        params = self.params

        training_data = os.path.join(self.config.data_ingestion.unzip_dir,"kedney_dataset")
        create_directorries([training.root_dir])

        trained_model_path = Path(training.trained_model_path)
        training_config = TrainingConfig(
            root_dir=Path(training.root_dir),
            trained_model_path=Path(training.trained_model_path),
            updated_base_model_path=Path(prepare_base_model.updated_base_model_path),
            training_data=Path(training_data),
            params_epochs=params.EPOCHS,
            params_batch_size= params.BATCH_SIZE,
            params_image_size = params.IMAGE_SIZE,
            params_is_augmentation = params.AUGMENTATION
        )

        return training_config



# AUGMENTATION: True
# IMAGE_SIZE: [224,224,3]
# BATCH_SIZE: 16
# INCLUDE_TOP: False
# EPOCHS: 1
# CLASSES: 2
# WEIGHTS: imagenet
# LEARNING_RATE: 0.01

In [ ]:
import os
import urllib.request as request
from zipfile import ZipFile
import tensorflow as tf
import time

class Training:
    def __init__(self,config:TrainingConfig):
        self.config = config
    def get_base_model(self):
        self.model = tf.keras.models.load_model(
            self.config.updated_base_model_path
        )
        # self.model.compile(
        #     optimizer=tf.keras.optimizers.SGD(learning_rate=0.01),
        #     loss=tf.keras.losses.CategoricalCrossentropy(),
        #     metrics=["accuracy"]
        # )
    def train_valid_generator(self):
        datagenerator_kwargs = dict(
            rescale = 1./255,
            validation_split=0.20
        )

        dataflow_kwargs = dict(
            target_size = self.config.params_image_size[:-1],
            batch_size=self.config.params_batch_size,
            interpolation="bilinear" 
        )

        valid_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
            **datagenerator_kwargs
        )

        self.valid_generator = valid_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset="validation",
            shuffle=False,
            **dataflow_kwargs
        )

        if self.config.params_is_augmentation:
            train_datagenerator = tf.keras.preprocessing.image.ImageDataGenerator(
                rotation_range=40,
                horizontal_flip=True,
                width_shift_range=0.2,
                height_shift_range=0.2,
                shear_range=0.2,
                zoom_range=0.2,
                **datagenerator_kwargs 
            )
        else:
            train_datagenerator = valid_datagenerator
        
        self.train_generator = train_datagenerator.flow_from_directory(
            directory=self.config.training_data,
            subset = "training",
            shuffle = True,
            **dataflow_kwargs
        )

    @staticmethod
    def save_model(path: Path, model: tf.keras.Model):
        model.save(path)

    def train(self):
        self.steps_per_epoch = self.train_generator.samples//self.train_generator.batch_size
        self.validation_steps = self.valid_generator.samples//self.valid_generator.batch_size

        self.model.fit(
            self.train_generator,
            epochs=self.config.params_epochs,
            steps_per_epoch=self.steps_per_epoch,
            validation_steps= self.validation_steps,
            validation_data= self.valid_generator
        )

        self.save_model(
            path=self.config.trained_model_path,
            model=self.model
        )


In [20]:
try:
    config = ConfigurationManger()
    training_config = config.get_training_config()
    training = Training(config=training_config)
    training.get_base_model()
    training.train_valid_generator()
    training.train()
except Exception as e:
    raise e

[2026-09-12 00:40:34,484: INFO : yaml file config\config.yaml loaded sucessfukky]
[2026-09-12 00:40:34,484: INFO : content {'artifacts_root': 'artifacts', 'data_ingestion': {'root_dir': 'artifacts/data_ingestion', 'source_URL': 'https://drive.google.com/file/d/1rL6uB6zH8wZWD1rB2yR4povMHWLukq0N/view?usp=sharing', 'local_data_file': 'artifacts/data_ingestion/data.zip', 'unzip_dir': 'artifacts/data_ingestion'}, 'prepare_base_model': {'root_dir': 'artifacts/prepare_base_model', 'base_model_path': 'artifacts/prepare_base_model/base_model.h5', 'updated_base_model_path': 'artifacts/prepare_base_model/base_model_updated.h5'}, 'training': {'root_dir': 'artifacts/training', 'trained_model_path': 'artifacts/training/model.h5'}}]
[2026-09-12 00:40:34,485: INFO : yaml file params.yaml loaded sucessfukky]
[2026-09-12 00:40:34,486: INFO : content {'AUGMENTATION': True, 'IMAGE_SIZE': [224, 224, 3], 'BATCH_SIZE': 16, 'INCLUDE_TOP': False, 'EPOCHS': 1, 'CLASSES': 4, 'WEIGHTS': 'imagenet', 'LEARNING_RATE

In [ ]:
%pwd

'd:\\Arambh\\project\\kedney\\Kidney-Disease-Classification'